# CORE-trm: Training

Training loop with:
- Mixed precision (AMP) for speed
- Early stopping on validation MRR@20
- Learning rate scheduling
- Gradient clipping
- 3-seed reproducibility
- Checkpointing

Note: CORE-trm is ~1.75x slower per epoch than CORE-ave due to Transformer.

In [ ]:
import os
import json
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from pathlib import Path
from typing import Dict, Any, Optional, Tuple
import numpy as np
from tqdm import tqdm

# Import model and dataset
import sys
sys.path.append("../../src")
from models.core_trm import CoreTrm, create_core_trm
from data.datasets import create_dataloaders, SessionDataset

In [ ]:
# Configuration
CONFIG = {
    # Model
    "vocab_size": 34644,
    "embedding_dim": 100,
    "max_seq_len": 50,
    "padding_idx": 0,
    "item_dropout": 0.2,
    "init_temperature": 1.0,
    "margin": 0.07,
    "min_temperature": 0.01,
    "n_layers": 2,
    "n_heads": 4,
    "ff_dim": 200,
    "transformer_dropout": 0.1,
    "attention_dropout": 0.1,
    
    # Training
    "batch_size": 2048,
    "learning_rate": 0.001,
    "weight_decay": 0.0,
    "beta1": 0.9,
    "beta2": 0.999,
    "eps": 1e-8,
    "max_epochs": 100,
    "patience": 5,
    "min_delta": 1e-4,
    "monitor_metric": "mrr@20",
    "use_scheduler": True,
    "scheduler_patience": 3,
    "scheduler_factor": 0.5,
    "scheduler_min_lr": 1e-5,
    "grad_clip": 1.0,
    "use_amp": True,
    "seeds": [42, 123, 456],
    "device": "cuda",
    "log_every_n_batches": 100,
    "save_best_only": True,
    
    # Paths
    "data_dir": "../../data/processed",
    "item_mapping": "../../data/processed/item_mapping.json",
    "checkpoint_dir": "../../checkpoints/core_trm",
    "logs_dir": "../../logs/core_trm",
}

In [ ]:
# Utility Functions
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device(device_str: str = "cuda") -> torch.device:
    if device_str == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

In [ ]:
# Create DataLoaders
train_loader, val_loader, test_loader, vocab_size = create_dataloaders(
    train_path=os.path.join(CONFIG["data_dir"], "train.parquet"),
    val_path=os.path.join(CONFIG["data_dir"], "validation.parquet"),
    test_path=os.path.join(CONFIG["data_dir"], "test.parquet"),
    item_mapping_path=CONFIG["item_mapping"],
    batch_size=CONFIG["batch_size"],
    max_len=CONFIG["max_seq_len"],
    num_workers=0,
    pin_memory=True
)

print(f"Vocab size: {vocab_size}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Training Functions
def train_one_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
    use_amp: bool = True,
    grad_clip: float = 1.0,
    log_every: int = 100
) -> Dict[str, float]:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    total_samples = 0
    batch_losses = []
    
    progress = tqdm(train_loader, desc="Training", leave=False)
    
    for batch_idx, (prefix, target, plen) in enumerate(progress):
        prefix = prefix.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        plen = plen.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if use_amp:
            with autocast():
                scores, loss, _ = model(prefix, plen, target)
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            scores, loss, _ = model(prefix, plen, target)
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        
        batch_size = prefix.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        batch_losses.append(loss.item())
        
        if (batch_idx + 1) % log_every == 0:
            progress.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / total_samples if total_samples > 0 else 0.0
    return {"train_loss": avg_loss, "batch_losses": batch_losses}

In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    val_loader: DataLoader,
    device: torch.device,
    k: int = 20,
    use_amp: bool = True
) -> Dict[str, float]:
    """Evaluate model on validation set."""
    model.eval()
    
    all_recall = []
    all_mrr = []
    total_samples = 0
    
    for prefix, target, plen in tqdm(val_loader, desc="Validation", leave=False):
        prefix = prefix.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        plen = plen.to(device, non_blocking=True)
        
        if use_amp:
            with autocast():
                scores, _, _ = model(prefix, plen, target)
        else:
            scores, _, _ = model(prefix, plen, target)
        
        topk = torch.topk(scores, k=k, dim=1).indices
        
        for i in range(prefix.size(0)):
            rec_list = topk[i].cpu().tolist()
            tgt = target[i].item()
            
            r = 1.0 if tgt in rec_list[:k] else 0.0
            
            try:
                rank = rec_list[:k].index(tgt) + 1
                m = 1.0 / rank
            except ValueError:
                m = 0.0
            
            all_recall.append(r)
            all_mrr.append(m)
        
        total_samples += prefix.size(0)
    
    return {
        "recall@20": float(np.mean(all_recall)),
        "mrr@20": float(np.mean(all_mrr)),
        "n_evaluated": total_samples
    }

In [ ]:
# Training Loop for Single Seed
def train_model(
    config: Dict[str, Any],
    seed: int,
    checkpoint_dir: Path,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader
) -> Dict[str, Any]:
    """Train a single model with given seed."""
    set_seed(seed)
    device = get_device(config["device"])
    
    print(f"\n{'='*60}")
    print(f"Training CORE-trm | Seed: {seed} | Device: {device}")
    print(f"{'='*60}")
    
    # Create model
    model = create_core_trm(
        vocab_size=config["vocab_size"],
        embedding_dim=config["embedding_dim"],
        padding_idx=config["padding_idx"],
        max_len=config["max_seq_len"],
        item_dropout=config["item_dropout"],
        init_temperature=config["init_temperature"],
        margin=config["margin"],
        min_temperature=config["min_temperature"],
        n_layers=config["n_layers"],
        n_heads=config["n_heads"],
        ff_dim=config["ff_dim"],
        transformer_dropout=config["transformer_dropout"],
        attention_dropout=config["attention_dropout"]
    ).to(device)
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        betas=(config["beta1"], config["beta2"]),
        eps=config["eps"],
        weight_decay=config["weight_decay"]
    )
    
    scheduler = None
    if config["use_scheduler"]:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=config["scheduler_factor"],
            patience=config["scheduler_patience"],
            min_lr=config["scheduler_min_lr"],
            verbose=True
        )
    
    scaler = GradScaler() if config["use_amp"] and device.type == "cuda" else None
    
    best_metric = -1.0
    best_epoch = 0
    epochs_no_improve = 0
    history = {
        "train_loss": [],
        "val_recall@20": [],
        "val_mrr@20": [],
        "lr": []
    }
    
    ckpt_path = checkpoint_dir / f"core_trm_seed{seed}_best.pt"
    
    for epoch in range(1, config["max_epochs"] + 1):
        epoch_start = time.time()
        
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, scaler, device,
            use_amp=config["use_amp"],
            grad_clip=config["grad_clip"],
            log_every=config["log_every_n_batches"]
        )
        
        val_metrics = evaluate(model, val_loader, device, k=20, use_amp=config["use_amp"])
        
        if scheduler is not None:
            scheduler.step(val_metrics[config["monitor_metric"]])
        
        history["train_loss"].append(train_metrics["train_loss"])
        history["val_recall@20"].append(val_metrics["recall@20"])
        history["val_mrr@20"].append(val_metrics["mrr@20"])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        
        current_metric = val_metrics[config["monitor_metric"]]
        improved = current_metric > best_metric + config["min_delta"]
        
        if improved:
            best_metric = current_metric
            best_epoch = epoch
            epochs_no_improve = 0
            
            if config["save_best_only"]:
                torch.save({
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "config": config,
                    "seed": seed,
                    "best_metric": best_metric,
                    "val_metrics": val_metrics,
                }, ckpt_path)
        else:
            epochs_no_improve += 1
        
        epoch_time = time.time() - epoch_start
        
        print(f"Epoch {epoch:3d}/{config['max_epochs']} | "
              f"Train Loss: {train_metrics['train_loss']:.4f} | "
              f"Val R@20: {val_metrics['recall@20']:.4f} | "
              f"Val MRR@20: {val_metrics['mrr@20']:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
              f"Time: {epoch_time:.1f}s"
              f"{'  <- BEST' if improved else ''}")
        
        if epochs_no_improve >= config["patience"]:
            print(f"Early stopping at epoch {epoch} (best at epoch {best_epoch})")
            break
    
    if ckpt_path.exists():
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"Loaded best model from epoch {checkpoint['epoch']}")
    
    print("\nEvaluating on test set...")
    test_metrics = evaluate(model, test_loader, device, k=20, use_amp=config["use_amp"])
    
    print(f"Test R@20: {test_metrics['recall@20']:.4f}")
    print(f"Test MRR@20: {test_metrics['mrr@20']:.4f}")
    
    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_metric": best_metric,
        "val_metrics": {
            "recall@20": history["val_recall@20"][best_epoch - 1] if best_epoch > 0 else 0,
            "mrr@20": history["val_mrr@20"][best_epoch - 1] if best_epoch > 0 else 0
        },
        "test_metrics": test_metrics,
        "history": history,
        "checkpoint": str(ckpt_path)
    }

In [ ]:
# Run Experiment with Multiple Seeds
import os
from pathlib import Path

checkpoint_dir = Path(CONFIG["checkpoint_dir"])
checkpoint_dir.mkdir(parents=True, exist_ok=True)

all_results = []

for seed in CONFIG["seeds"]:
    result = train_model(CONFIG, seed, checkpoint_dir, train_loader, val_loader, test_loader)
    all_results.append(result)
    
    results_path = checkpoint_dir / f"results_seed{seed}.json"
    with open(results_path, 'w') as f:
        json.dump(result, f, indent=2, default=str)

# Aggregate statistics
test_recalls = [r["test_metrics"]["recall@20"] for r in all_results]
test_mrrs = [r["test_metrics"]["mrr@20"] for r in all_results]
val_recalls = [r["val_metrics"]["recall@20"] for r in all_results]
val_mrrs = [r["val_metrics"]["mrr@20"] for r in all_results]

summary = {
    "model_type": "core_trm",
    "config": CONFIG,
    "seeds": CONFIG["seeds"],
    "individual_results": all_results,
    "aggregate": {
        "test_recall@20": {
            "mean": float(np.mean(test_recalls)),
            "std": float(np.std(test_recalls)),
            "values": test_recalls
        },
        "test_mrr@20": {
            "mean": float(np.mean(test_mrrs)),
            "std": float(np.std(test_mrrs)),
            "values": test_mrrs
        },
        "val_recall@20": {
            "mean": float(np.mean(val_recalls)),
            "std": float(np.std(val_recalls)),
            "values": val_recalls
        },
        "val_mrr@20": {
            "mean": float(np.mean(val_mrrs)),
            "std": float(np.std(val_mrrs)),
            "values": val_mrrs
        }
    }
}

summary_path = checkpoint_dir / "summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n{'='*60}")
print(f"EXPERIMENT COMPLETE: core_trm")
print(f"{'='*60}")
print(f"Test Recall@20: {summary['aggregate']['test_recall@20']['mean']:.4f} ± {summary['aggregate']['test_recall@20']['std']:.4f}")
print(f"Test MRR@20:    {summary['aggregate']['test_mrr@20']['mean']:.4f} ± {summary['aggregate']['test_mrr@20']['std']:.4f}")
print(f"Results saved to: {summary_path}")

In [ ]:
# Plot Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for seed in CONFIG["seeds"]:
    result_path = checkpoint_dir / f"results_seed{seed}.json"
    with open(result_path) as f:
        result = json.load(f)
    
    history = result["history"]
    epochs = range(1, len(history["train_loss"]) + 1)
    
    axes[0, 0].plot(epochs, history["train_loss"], label=f'Seed {seed}')
    axes[0, 1].plot(epochs, history["val_recall@20"], label=f'Seed {seed}')
    axes[1, 0].plot(epochs, history["val_mrr@20"], label=f'Seed {seed}')
    axes[1, 1].plot(epochs, history["lr"], label=f'Seed {seed}')

axes[0, 0].set_title("Training Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].set_title("Validation Recall@20")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Recall@20")
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].set_title("Validation MRR@20")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("MRR@20")
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].set_title("Learning Rate")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("LR")
axes[1, 1].legend()
axes[1, 1].grid(True)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig("../../reports/models/core_trm_training_curves.png", dpi=150, bbox_inches='tight')
plt.close()
print("Training curves saved")